# Summarizing Text and PDF Data with CatLLM

CatLLM provides a unified `cat.summarize()` function for summarizing open-ended text data (such as survey responses) and PDF documents using large language models.

**What this notebook covers:**
- Basic single-model text summarization
- Switching between providers (OpenAI, Anthropic, Google, etc.)
- Controlling summary output (length, focus, instructions)
- Prompt strategies (chain-of-thought, step-back, context prompt)
- Robustness features: safety saves, row delay, retry controls, fail strategy
- Batch mode for 50% cost savings
- Multi-model ensemble summarization with consensus synthesis
- PDF summarization

**What you need:**
- Python 3.9+
- `pip install catllm` (or `pip install catllm[pdf]` for PDF support)
- An API key from at least one supported provider

**How it works:**

For each text or PDF page, CatLLM sends a prompt to the LLM asking it to generate a summary. The output is a DataFrame with a `summary` column for each input. Input type (text vs PDF) is auto-detected from your data.

In [ ]:
import catllm as cat
import pandas as pd

## Setup

Set your API key for the provider you want to use. CatLLM supports OpenAI, Anthropic, Google, Mistral, Perplexity, xAI, HuggingFace, and Ollama (local).

In [ ]:
# Replace with your actual API key for your chosen provider
api_key = "YOUR_API_KEY"

## Sample Data

Here we create some sample survey responses to summarize. These could be open-ended answers, customer feedback, interview transcripts, or any free-text data.

In [ ]:
# Open-ended survey responses to summarize
responses = [
    "I moved because my company relocated the entire office to Austin. I didn't really "
    "want to leave Chicago but the job was too good to pass up, and they offered a "
    "relocation package that covered most of the costs.",
    
    "We needed a bigger house for our growing family. Our second child was on the way "
    "and the two-bedroom apartment just wasn't going to cut it anymore. We found a "
    "nice place in the suburbs with a yard.",
    
    "The rent kept going up every year and we just couldn't afford it anymore. We were "
    "spending over 40% of our income on housing. We moved to a smaller city where the "
    "cost of living is much more reasonable.",
    
    "After my divorce I had to find a new place. I moved closer to my parents so they "
    "could help with the kids. It was a really difficult time but having family nearby "
    "made a huge difference.",
    
    "I retired last year and my wife and I decided to move somewhere warmer. We sold "
    "our house in Minnesota and bought a condo in Arizona. We love the weather and "
    "there's a great community of retirees here.",
]

## Basic Summarization

The simplest usage requires just two things:
1. `input_data` — the text you want to summarize (a list, pandas Series, or single string)
2. `api_key` — your API key

CatLLM auto-detects the provider from the model name.

In [ ]:
result = cat.summarize(
    input_data=responses,
    user_model="gpt-4o-mini",
    api_key=api_key,
)

result.head()

## Adding Context and Instructions

Providing a `description` gives the model context about what it's summarizing. You can also pass `instructions` to control the output format, `max_length` to cap the word count, and `focus` to steer what the summary emphasizes.

In [ ]:
result_context = cat.summarize(
    input_data=responses,
    description="Open-ended survey responses about why people moved to their current residence",
    instructions="Summarize in bullet points",
    max_length=50,
    focus="the primary motivation for moving",
    user_model="gpt-4o-mini",
    api_key=api_key,
)

result_context.head()

## Switching Providers

CatLLM auto-detects the provider from the model name. Just swap the model and API key.

In [ ]:
# Anthropic — auto-detected from model name
result_anthropic = cat.summarize(
    input_data=responses,
    description="Survey responses about why people moved",
    user_model="claude-sonnet-4-20250514",
    api_key="ANTHROPIC_API_KEY",
)

result_anthropic.head()

## Prompt Strategies

CatLLM supports several prompting strategies that can improve summary quality:

- **`chain_of_thought=True`** (default for summarize): Asks the model to reason step-by-step before summarizing. On by default.
- **`context_prompt=True`**: Adds an expert analyst role to the prompt.
- **`step_back_prompt=True`**: Asks the model to consider the broader context before summarizing. This makes an extra API call per model to gather "step-back insights."

In [ ]:
result_prompted = cat.summarize(
    input_data=responses,
    description="Survey responses about why people moved",
    chain_of_thought=True,
    context_prompt=True,
    step_back_prompt=True,
    user_model="gpt-4o-mini",
    api_key=api_key,
)

result_prompted.head()

## Safety: Incremental Saves

For large datasets, enable `safety=True` to save progress to a CSV file after each row is processed. If the process crashes or is interrupted midway, you won't lose your work.

Requires a `filename` to be set.

In [ ]:
result_safe = cat.summarize(
    input_data=responses,
    description="Survey responses about why people moved",
    user_model="gpt-4o-mini",
    api_key=api_key,
    safety=True,
    filename="summarized_results.csv",
    save_directory="./output",
)

result_safe.head()

## Row Delay

Use `row_delay` to add a pause (in seconds) between processing each row. This is useful when working with providers that have strict rate limits, or when multiple models in an ensemble share the same API key.

In [ ]:
result_delayed = cat.summarize(
    input_data=responses,
    description="Survey responses about why people moved",
    user_model="gpt-4o-mini",
    api_key=api_key,
    row_delay=1.0,  # 1 second pause between rows
)

result_delayed.head()

## Retry Controls

CatLLM provides fine-grained control over how failures are handled:

- **`max_retries`**: Max retries per individual API call (default 5)
- **`batch_retries`**: After all rows are processed, retry failed (row, model) pairs this many times (default 2)
- **`retry_delay`**: Seconds to wait between retry passes (default 1.0)
- **`fail_strategy`**: `"partial"` (default) keeps successful results even if some models fail; `"strict"` blanks the entire row if any model fails

In [ ]:
result_retries = cat.summarize(
    input_data=responses,
    description="Survey responses about why people moved",
    user_model="gpt-4o-mini",
    api_key=api_key,
    max_retries=3,
    batch_retries=1,
    retry_delay=2.0,
    fail_strategy="strict",
)

result_retries.head()

## Batch Mode (50% Cost Savings)

For large datasets, `batch_mode=True` submits all rows as an async batch job. Batch API pricing is typically 50% cheaper and has higher rate limits. The tradeoff is latency — batch jobs can take minutes to hours to complete.

Supported providers: OpenAI, Anthropic, Google, Mistral, xAI.

**Note:** Batch mode is text-only — it does not support PDF input.

In [ ]:
result_batch = cat.summarize(
    input_data=responses,
    description="Survey responses about why people moved",
    user_model="gpt-4o-mini",
    api_key=api_key,
    batch_mode=True,
    batch_poll_interval=15.0,  # Check job status every 15 seconds
)

result_batch.head()

## Multi-Model Ensemble

Pass a `models` list of `(model, provider, api_key)` tuples to summarize with multiple models simultaneously. CatLLM collects all model summaries and then synthesizes them into a single consensus summary using an LLM.

The output includes:
- `summary` — the synthesized consensus summary
- `summary_<model>` — each individual model's summary
- `failed_models` — which models (if any) failed for each row

In [ ]:
result_ensemble = cat.summarize(
    input_data=responses,
    description="Survey responses about why people moved",
    models=[
        ("gpt-4o-mini", "openai", "OPENAI_API_KEY"),
        ("claude-sonnet-4-20250514", "anthropic", "ANTHROPIC_API_KEY"),
    ],
)

result_ensemble.head()

## Multi-Model Batch Mode

Combine `batch_mode=True` with `models` for maximum cost savings. Each model submits its own batch job concurrently. Providers without a batch API (HuggingFace, Perplexity, Ollama) automatically fall back to synchronous calls.

In [ ]:
result_ensemble_batch = cat.summarize(
    input_data=responses,
    description="Survey responses about why people moved",
    models=[
        ("gpt-4o-mini", "openai", "OPENAI_API_KEY"),
        ("claude-sonnet-4-20250514", "anthropic", "ANTHROPIC_API_KEY"),
    ],
    batch_mode=True,
)

result_ensemble_batch.head()

## PDF Summarization

CatLLM auto-detects PDF input from file paths. Pass a directory, a single PDF path, or a list of PDF paths. The `mode` parameter controls how pages are processed:

- **`"image"`** (default): Render each page as an image — best for visual documents
- **`"text"`**: Extract text only — faster, good for text-heavy PDFs
- **`"both"`**: Send both image and extracted text — most comprehensive

Requires: `pip install catllm[pdf]`

In [ ]:
# Summarize all PDFs in a directory
result_pdf = cat.summarize(
    input_data="/path/to/pdfs/",
    description="Research papers on climate change",
    mode="image",
    max_length=100,
    user_model="gpt-4o",
    api_key=api_key,
)

result_pdf.head()

## Combining Features

All parameters can be combined. Here's an example using safety saves, row delay, retry controls, and prompt strategies together.

In [ ]:
result_combined = cat.summarize(
    input_data=responses,
    description="Survey responses about why people moved",
    instructions="Write a one-sentence summary",
    focus="the primary reason for moving",
    max_length=30,
    user_model="gpt-4o-mini",
    api_key=api_key,
    # Prompt strategies
    chain_of_thought=True,
    context_prompt=True,
    # Robustness
    safety=True,
    filename="combined_results.csv",
    save_directory="./output",
    row_delay=0.5,
    max_retries=3,
    batch_retries=1,
    retry_delay=1.0,
    fail_strategy="partial",
)

result_combined.head()

## Summary: Parameter Reference

| Parameter | Default | Effect |
|-----------|---------|--------|
| `api_key` | `None` | API key for the provider (single-model mode). |
| `user_model` | `"gpt-4o"` | Model to use. Provider is auto-detected from the name. |
| `model_source` | `"auto"` | Provider override — `"auto"`, `"openai"`, `"anthropic"`, `"ollama"`, etc. |
| `description` | `""` | Context about the content being summarized. |
| `instructions` | `""` | Specific format instructions (e.g., "bullet points"). |
| `max_length` | `None` | Maximum summary length in words. |
| `focus` | `None` | What to emphasize (e.g., "key arguments"). |
| `creativity` | `None` | Temperature (0.0–1.0). `None` uses model default. |
| `chain_of_thought` | `True` | Step-by-step reasoning before summarizing. |
| `context_prompt` | `False` | Add expert analyst role to the prompt. |
| `step_back_prompt` | `False` | Broader reasoning before summarizing. Extra API call. |
| `safety` | `False` | Save progress after each row. Requires `filename`. |
| `filename` | `None` | Output CSV filename. Required for `safety=True`. |
| `save_directory` | `None` | Directory to save results. |
| `row_delay` | `0.0` | Seconds to pause between processing each row. |
| `max_retries` | `5` | Max retries per individual API call. |
| `batch_retries` | `2` | Retry passes for failed (row, model) pairs. |
| `retry_delay` | `1.0` | Seconds between retry passes. |
| `fail_strategy` | `"partial"` | `"partial"` keeps successes; `"strict"` blanks row on any failure. |
| `batch_mode` | `False` | Async batch processing. 50% cheaper, higher latency. Text only. |
| `batch_poll_interval` | `30.0` | Seconds between batch job status checks. |
| `batch_timeout` | `86400` | Max seconds to wait for batch job (default 24h). |
| `mode` | `"image"` | PDF processing mode: `"image"`, `"text"`, or `"both"`. |
| `models` | `None` | List of `(model, provider, api_key)` tuples for ensemble mode. |

**Supported providers:** OpenAI, Anthropic, Google, Mistral, Perplexity, xAI, HuggingFace, Ollama (local).